In [2]:
from calculos_plasmas import *
import matplotlib.pyplot as plt

def analizar_autovalores(plasma : Plasma, titulo="Comparación de Autovalores"):
    """
    Compara las energías numéricas de un objeto Plasma con las analíticas del hidrógeno.
    """
    if plasma.E_0 is None:
        raise ValueError("Todavía no se han computado los autovalores.")

    n_values = [1]
    E_num = [plasma.E_0]
    E_ana = [-0.5*E_h*plasma.Z**2]

    for l in range(1, len(plasma.E_l)):
        E_l = plasma.E_l[l]
        n_start = l + 1 if l > 0 else 2

        for i, E_nl in enumerate(E_l):
            n = n_start + i
            n_values.append(n)
            E_num.append(E_nl)
            E_ana.append(-0.5*E_h*plasma.Z**2 / n**2)

    n_values = np.array(n_values)
    E_num_eV = np.array(E_num) / E_h
    E_ana_eV = np.array(E_ana) / E_h

    mse_eV = np.mean((E_num_eV - E_ana_eV)**2)

    plt.figure(figsize=(10, 6))
    plt.plot(n_values, E_ana_eV, 'kx', markersize=10, label='Analítico')
    plt.plot(n_values, E_num_eV, 'ro', alpha=0.7, label=f'Numérico')

    plt.title(f"{titulo}\nDistancia Cuadrática Media: {mse_eV:.2e} $eV^2$")
    plt.xlabel("Número cuántico principal ($n$)")
    plt.ylabel("Energía (eV)")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.show()

    return mse_eV

In [3]:
T = [1, 5, 1, 5, 1, 5]
n = [1e18, 1e18, 1e20, 1e20, 1e22, 1e22]
n_max = [10, 10, 5, 5, 1, 1]
C = []
for i in range(6):
    P = Plasma()
    P.set_temperature(T[i], 'eV')
    P.set_electron_density(n[i], "cm")
    P.Z = 1
    P.n_max = n_max[i]
    P.l_max = n_max[i]-1
    print(f"Resolviendo plasma a {T[i]} eV y {n[i]} cm^-3")
    P.solve_schrodinger()
    if P.E_0 is None:
        print("¡Aviso! No se obtuvieron autovalores")
        C.append(None)
        continue
    P.solve_population()
    C.append(round(float(P.n_H/P.n_P), 3))
expected_C = [231.56, 0.012, 7691, 0.082, 2324, 1.32]
print(C)
print(expected_C)

Resolviendo plasma a 1 eV y 1e+18 cm^-3
Resolviendo plasma a 5 eV y 1e+18 cm^-3
Resolviendo plasma a 1 eV y 1e+20 cm^-3
Resolviendo plasma a 5 eV y 1e+20 cm^-3
Resolviendo plasma a 1 eV y 1e+22 cm^-3
Resolviendo plasma a 5 eV y 1e+22 cm^-3
[206.322, 0.012, 771.376, 0.181, 3.743, 0.515]
[231.56, 0.012, 7691, 0.082, 2324, 1.32]


Empezamos con un plasma sin efectos de apantallamiento en el potencial

In [4]:
A = Plasma()
A.set_temperature(1, "eV")
A.set_electron_density(1e18, "cm")
A.set_atomic_mesh(2e5)
A.max_radius = 500
A.n_max = 10
A.l_max = A.n_max-1
A.Z = 1
A.solve_plasma(plasma_effects=True)

Resolviendo ecuación de autovalores...
Resolviendo transiciones...
Resolviendo poblaciones...
Resolviendo espectro...


In [6]:
print((A.n_P/A.n_H)/0.00186)

2.60580791666854


In [ ]:
analizar_autovalores(A)

Comprobemos ahora que hemos elegido correctamente el radio máximo

In [ ]:
def plot_frontera(plasma, l, n):
    if A.n_max == 1 or len(A.E_l[l]+l+1) < n:
        raise ValueError("Error: En el plasma no existe el estado deseado")

    u = plasma.u_l[l]/E_h

    i = n - l - 1
    if i < 0 or i >= u.shape[1]:
        raise NameError(f"Error: El estado n={n} no existe físicamente para l={l} con los parámetros actuales.")

    u_nl = u[:, i]

    plt.figure(figsize=(8, 4))
    plt.plot(plasma.r, u_nl, label=f'Densidad $u(r)$ para $n={n}$')
    plt.axvline(100, color='red', linestyle='--', label='Radio clásico (100 u.a.)')

    plt.title(f'Comprobación de la frontera para $l={l}, n={n}$')
    plt.xlabel(r'$r$ (unidades atómicas)')
    plt.ylabel('Densidad de probabilidad')
    plt.xlim(0, plasma.max_radius)
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
plot_frontera(A, 0, len(A.E_l[0]))

Ahora compararemos los valores numéricos con los analíticos en la transición

In [ ]:
n = np.arange(2, len(A.nu_if)+2)
nu_analitic = c * Rydberg * (1 - 1 / n**2)
print(A.nu_if)
print(nu_analitic)
print(f"Error cuadrático medio: {np.mean((A.nu_if-nu_analitic)**2): .2e}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))

ax1.plot(A.nu, A.I, color='crimson', linewidth=1.5)
ax1.set_title("Espectro de Emisión: Serie Lyman")
ax1.set_xlabel("Frecuencia $\\nu$ (Hz)")
ax1.set_ylabel("Intensidad Específica I)($\\nu$) (S.I.)")
ax1.grid(True, alpha=0.3)

ax2.plot(A.nu, A.I, color='crimson', linewidth=1.5)
ax2.set_title(r"Espectro de Emisión: Línea Lyman-$\alpha$")
ax2.set_xlabel(r"Frecuencia $\nu$ (Hz)")
ax2.set_ylabel(r"Intensidad Específica I)($\nu$) (S.I.)")
ax2.grid(True, alpha=0.3)
frecuencia_lyman_alfa = A.nu_if[0]
ancho_zoom = 5e12
ax2.set_xlim(frecuencia_lyman_alfa - ancho_zoom, frecuencia_lyman_alfa + ancho_zoom)